In [0]:
# =============================================================================
# jobs/07_compute_intersections.py
# Compute FGS polygon intersections against EA flood areas and constituencies.
#
# What this script does:
#   1. Identifies FGS statements not yet intersected (or all, if recompute_all).
#   2. For each statement, for each FGS source:
#        a. Filters EA areas to only those relevant for that source.
#        b. Intersects FGS polygons against matching EA areas (25% threshold).
#           Appends results to fgs_ea_area_intersections.
#        c. Intersects FGS polygons against constituencies (any overlap).
#           Appends results to fgs_constituency_intersections.
#
# FGS source to EA type_code mapping:
#   river   -> FWF, WAF, FWB, WAB
#   coastal -> FWC, WAC, FWB, WAB, FWT, WAT
#   ground  -> FWG, FAG
#   surface -> no EA equivalent, skipped
#
# Risk matrix columns carried through from fgs_risk_polygons:
#   risk_x           -- impact coordinate (1=Minimal to 4=Severe)
#   risk_y           -- likelihood coordinate (1=Very Low to 4=High)
#   impact_label     -- human-readable impact label
#   likelihood_label -- human-readable likelihood label
#   risk_level       -- overall risk (Very Low / Low / Medium / High)
#   risk_colour      -- traffic light colour (Green / Yellow / Amber / Red)
# =============================================================================

import sys
import json

sys.path.insert(0, "/Workspace/Users/jon.payne@environment-agency.gov.uk/FGS_Notebooks/")
from config import (
    EA_INTERSECTION_MIN_PCT,
    TBL_RISK_POLYGONS, TBL_EA_FWA, TBL_EA_FAA,
    TBL_CONSTITUENCIES, TBL_FGS_EA_INTERSECT, TBL_FGS_CONST_INTERSECT
)
from utils.helpers import get_spark, utc_now, table_exists

import geopandas as gpd
import pandas as pd
from shapely.geometry import shape
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType
)
from pyspark.sql.functions import col


# =============================================================================
# FGS SOURCE TO EA TYPE CODE MAPPING
# =============================================================================

FGS_SOURCE_TYPE_CODES = {
    "river":   {"FWF", "WAF", "FWB", "WAB"},
    "coastal": {"FWC", "WAC", "FWB", "WAB", "FWT", "WAT"},
    "ground":  {"FWG", "FAG"},
    "surface": set(),   # No EA equivalent -- skip entirely
}


# =============================================================================
# SETUP
# =============================================================================

spark   = get_spark()
now_iso = utc_now().isoformat()

try:
    recompute_all = dbutils.widgets.get("recompute_all").lower() == "true"
except Exception:
    recompute_all = False

print(f"recompute_all = {recompute_all}")

In [0]:
# =============================================================================
# SCHEMAS
# =============================================================================

ea_intersect_schema = StructType([
    StructField("statement_id",      IntegerType(), True),
    StructField("issued_at",         StringType(),  True),
    StructField("poly_id",           IntegerType(), True),
    StructField("day_index",         IntegerType(), True),
    StructField("forecast_date",     StringType(),  True),
    StructField("ea_area_code",      StringType(),  True),
    StructField("ea_area_type",      StringType(),  True),
    StructField("ea_area_name",      StringType(),  True),
    StructField("source",            StringType(),  True),
    StructField("risk_x",            IntegerType(), True),
    StructField("risk_y",            IntegerType(), True),
    StructField("impact_label",      StringType(),  True),
    StructField("likelihood_label",  StringType(),  True),
    StructField("risk_level",        StringType(),  True),
    StructField("risk_colour",       StringType(),  True),
    StructField("intersection_pct",  FloatType(),   True),
    StructField("computed_at",       StringType(),  True),
])

const_intersect_schema = StructType([
    StructField("statement_id",      IntegerType(), True),
    StructField("issued_at",         StringType(),  True),
    StructField("poly_id",           IntegerType(), True),
    StructField("day_index",         IntegerType(), True),
    StructField("forecast_date",     StringType(),  True),
    StructField("constituency_id",   StringType(),  True),
    StructField("constituency_name", StringType(),  True),
    StructField("source",            StringType(),  True),
    StructField("risk_x",            IntegerType(), True),
    StructField("risk_y",            IntegerType(), True),
    StructField("impact_label",      StringType(),  True),
    StructField("likelihood_label",  StringType(),  True),
    StructField("risk_level",        StringType(),  True),
    StructField("risk_colour",       StringType(),  True),
    StructField("intersection_pct",  FloatType(),   True),
    StructField("computed_at",       StringType(),  True),
])


In [0]:
# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def to_geodataframe(pdf: pd.DataFrame, crs: str = "EPSG:4326") -> gpd.GeoDataFrame:
    """
    Convert a Pandas DataFrame with a GeoJSON geometry string column
    into a GeoPandas GeoDataFrame.
    Handles None, NaN, and non-string values gracefully.
    """
    def parse_geom(s):
        if s is None or not isinstance(s, str):
            return None
        try:
            return shape(json.loads(s))
        except Exception:
            return None

    geometries = pdf["geometry"].apply(parse_geom)
    return gpd.GeoDataFrame(pdf, geometry=geometries, crs=crs)


def clean_geodataframe(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Repair invalid geometries using buffer(0).
    Works on all GEOS versions. Drops unrepairable rows.
    """
    def fix_geom(g):
        if g is None:
            return None
        try:
            if g.is_valid:
                return g
            return g.buffer(0)
        except Exception:
            return g

    gdf = gdf.copy()
    gdf["geometry"] = gdf["geometry"].apply(fix_geom)
    gdf = gdf[gdf["geometry"].notna()].reset_index(drop=True)
    return gdf


In [0]:
# =============================================================================
# IDENTIFY STATEMENTS TO PROCESS
# =============================================================================

risk_df = spark.table(TBL_RISK_POLYGONS)

if recompute_all or not table_exists(spark, TBL_FGS_EA_INTERSECT):
    statements_to_process = [
        r["statement_id"]
        for r in risk_df.select("statement_id").distinct().collect()
    ]
    print(f"Processing all {len(statements_to_process)} statements.")
else:
    already_done = (
        spark.table(TBL_FGS_EA_INTERSECT)
        .select("statement_id").distinct()
    )
    statements_to_process = [
        r["statement_id"]
        for r in risk_df.select("statement_id").distinct()
        .subtract(already_done)
        .collect()
    ]
    print(f"{len(statements_to_process)} new statements to process.")

if not statements_to_process:
    print("Nothing to process. Exiting.")
    dbutils.notebook.exit("no_work")


In [0]:
# =============================================================================
# LOAD AND PREPARE REFERENCE GEOMETRIES ONCE
# =============================================================================

print("Loading reference geometries...")

fwa_pdf = spark.table(TBL_EA_FWA).toPandas()
faa_pdf = spark.table(TBL_EA_FAA).toPandas()
ea_pdf  = pd.concat([fwa_pdf, faa_pdf], ignore_index=True)
ea_gdf  = to_geodataframe(ea_pdf)

const_pdf = spark.table(TBL_CONSTITUENCIES).toPandas()
const_gdf = to_geodataframe(const_pdf)

ea_bng    = ea_gdf.to_crs("EPSG:27700")
const_bng = const_gdf.to_crs("EPSG:27700")
ea_bng    = clean_geodataframe(ea_bng)
const_bng = clean_geodataframe(const_bng)

ea_geom_lookup    = dict(zip(ea_bng["ea_area_code"],       ea_bng["geometry"]))
const_geom_lookup = dict(zip(const_bng["constituency_id"], const_bng["geometry"]))

print(f"  {len(ea_bng)} EA areas, {len(const_bng)} constituencies ready.")


In [0]:
# =============================================================================
# PROCESS EACH STATEMENT
# =============================================================================

for statement_id in statements_to_process:
    print(f"Processing statement_id: {statement_id}")

    stmt_pdf = (
        risk_df
        .filter(col("statement_id") == statement_id)
        .toPandas()
    )

    if stmt_pdf.empty:
        print(f"  No risk polygons -- skipping.")
        continue

    stmt_gdf = to_geodataframe(stmt_pdf)
    stmt_bng = stmt_gdf.to_crs("EPSG:27700")
    stmt_bng = clean_geodataframe(stmt_bng)

    if stmt_bng.empty:
        print(f"  All polygons invalid after cleaning -- skipping.")
        continue

    # -------------------------------------------------------------------------
    # EA FLOOD AREA INTERSECTIONS
    # -------------------------------------------------------------------------

    ea_rows = []
    sources = stmt_bng["source"].unique()

    for source in sources:
        relevant_codes = FGS_SOURCE_TYPE_CODES.get(source, set())

        if not relevant_codes:
            print(f"  Skipping source '{source}' -- no EA area equivalent.")
            continue

        source_bng    = stmt_bng[stmt_bng["source"] == source]
        ea_source_bng = ea_bng[ea_bng["type_code"].isin(relevant_codes)]

        if ea_source_bng.empty:
            continue

        ea_joined = gpd.sjoin(
            ea_source_bng,
            source_bng[["poly_id", "day_index", "forecast_date",
                        "source", "risk_x", "risk_y",
                        "impact_label", "likelihood_label",
                        "risk_level", "risk_colour",
                        "issued_at", "geometry"]],
            how="inner",
            predicate="intersects"
        )

        if len(ea_joined) == 0:
            continue

        stmt_geom_lookup = dict(zip(source_bng["poly_id"], source_bng["geometry"]))

        for _, row in ea_joined.iterrows():
            ea_geom   = ea_geom_lookup.get(row.get("ea_area_code"))
            poly_geom = stmt_geom_lookup.get(row.get("poly_id"))

            try:
                pct = ea_geom.intersection(poly_geom).area / ea_geom.area
            except Exception:
                pct = 0.0

            if pct < EA_INTERSECTION_MIN_PCT:
                continue

            ea_rows.append(Row(
                statement_id     = int(statement_id),
                issued_at        = str(row.get("issued_at")        or ""),
                poly_id          = int(row.get("poly_id")          or 0),
                day_index        = int(row.get("day_index")        or 0),
                forecast_date    = str(row.get("forecast_date")    or ""),
                ea_area_code     = str(row.get("ea_area_code")     or ""),
                ea_area_type     = str(row.get("ea_area_type")     or ""),
                ea_area_name     = str(row.get("ea_area_name")     or ""),
                source           = str(source),
                risk_x           = int(row.get("risk_x")           or 0),
                risk_y           = int(row.get("risk_y")           or 0),
                impact_label     = str(row.get("impact_label")     or ""),
                likelihood_label = str(row.get("likelihood_label") or ""),
                risk_level       = str(row.get("risk_level")       or ""),
                risk_colour      = str(row.get("risk_colour")      or ""),
                intersection_pct = float(pct),
                computed_at      = now_iso
            ))

    if ea_rows:
        spark.createDataFrame(ea_rows, schema=ea_intersect_schema).write \
            .format("delta").mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(TBL_FGS_EA_INTERSECT)
        print(f"  Written {len(ea_rows)} EA intersection rows.")
    else:
        print(f"  No EA intersections met the {EA_INTERSECTION_MIN_PCT*100:.0f}% threshold.")

    # -------------------------------------------------------------------------
    # CONSTITUENCY INTERSECTIONS
    # -------------------------------------------------------------------------

    const_joined = gpd.sjoin(
        const_bng,
        stmt_bng[["poly_id", "day_index", "forecast_date",
                  "source", "risk_x", "risk_y",
                  "impact_label", "likelihood_label",
                  "risk_level", "risk_colour",
                  "issued_at", "geometry"]],
        how="inner",
        predicate="intersects"
    )

    const_rows = []

    if len(const_joined) > 0:
        stmt_geom_lookup = dict(zip(stmt_bng["poly_id"], stmt_bng["geometry"]))

        for _, row in const_joined.iterrows():
            const_geom = const_geom_lookup.get(row.get("constituency_id"))
            poly_geom  = stmt_geom_lookup.get(row.get("poly_id"))

            try:
                pct = const_geom.intersection(poly_geom).area / const_geom.area
            except Exception:
                pct = 0.0

            const_rows.append(Row(
                statement_id     = int(statement_id),
                issued_at        = str(row.get("issued_at")        or ""),
                poly_id          = int(row.get("poly_id")          or 0),
                day_index        = int(row.get("day_index")        or 0),
                forecast_date    = str(row.get("forecast_date")    or ""),
                constituency_id  = str(row.get("constituency_id")  or ""),
                constituency_name= str(row.get("name")             or ""),
                source           = str(row.get("source")           or ""),
                risk_x           = int(row.get("risk_x")           or 0),
                risk_y           = int(row.get("risk_y")           or 0),
                impact_label     = str(row.get("impact_label")     or ""),
                likelihood_label = str(row.get("likelihood_label") or ""),
                risk_level       = str(row.get("risk_level")       or ""),
                risk_colour      = str(row.get("risk_colour")      or ""),
                intersection_pct = float(pct),
                computed_at      = now_iso
            ))

    if const_rows:
        spark.createDataFrame(const_rows, schema=const_intersect_schema).write \
            .format("delta").mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable(TBL_FGS_CONST_INTERSECT)
        print(f"  Written {len(const_rows)} constituency intersection rows.")
    else:
        print(f"  No constituency intersections found for statement {statement_id}.")

print("Intersection computation complete.")
dbutils.notebook.exit("success")
